In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import math
import glob
import re

In [2]:
votes_2022_df=pd.read_csv('./data/voter_numbers_2022_with_alt_map.csv')
votes_2022_df['indep_votes'] = votes_2022_df['indep_votes'].replace(np.nan, 0).astype(int)
votes_2022_df['total_votes'] =  votes_2022_df[['dem_votes', 'rep_votes']].sum(axis=1)
votes_2022_df

,county_id,NAME10,dem_votes,rep_votes,indep_votes,og_district,alt_district,total_votes
0,0,Adair,973,2166,0,3,4,3139
1,1,Adams,510,1126,0,3,3,1636
2,2,Allamakee,1932,3820,0,2,2,5752
3,3,Appanoose,1412,3249,0,3,3,4661
4,4,Audubon,633,1639,51,4,4,2272
...,...,...,...,...,...,...,...,...
94,94,Winnebago,1303,2924,101,4,4,4227
95,95,Winneshiek,4445,5135,0,2,2,9580
96,96,Woodbury,9601,18234,533,4,4,27835
97,97,Worth,1189,2031,0,2,4,3220


In [3]:
df=pd.read_csv('./allocation_by_county/seed_000_by_county.csv')
df.head()

,COUNTYFP10,NAME10,geometry,population,county_id,xcentr_lon,ycentr_lat,county_id_string,temp_district,num_switches,DISTRICT
0,1,Adair,"POLYGON ((386139.6726560561 4557123.147154345,...",7496,0,376909.476792,4.576509e+06,0,0,0,2
1,3,Adams,"POLYGON ((347722.4636164992 4557927.937157947,...",3704,1,357154.753953,4.543364e+06,1,1,1,3
2,5,Allamakee,"POLYGON ((621990.1817773253 4817526.586308301,...",14061,2,631594.469070,4.793646e+06,2,2,0,4
3,7,Appanoose,POLYGON ((491767.0579541979 4517888.7622878505...,12317,3,511092.713138,4.510250e+06,3,3,0,3
4,9,Audubon,POLYGON ((325801.57805079524 4617291.998950814...,5674,4,341387.505752,4.616499e+06,4,4,0,1


In [ ]:
# first argument is the voter number dataframe, second argument is the simulation dataframe with new district assignments
# output is total democrat and republican votes in each district, along with the winning number based on votes cast in district
def district_totals(votes_df, realloc_df):
    votes_df['simulation_district']=realloc_df['DISTRICT']
    district_votes=pd.DataFrame(columns=['district', 'dem_votes', 'rep_votes', 'votes_to_win'])
    for i in range(1,5):
        district_total= votes_df.loc[votes_df['simulation_district']==i]['total_votes'].sum()
        votes_to_win = 0
        if district_total % 2 ==0:
            votes_to_win = votes_to_win+(district_total/2) + 1
        else:
            votes_to_win = votes_to_win+math.ceil(district_total/2)
        district_votes.loc[i]=[i,
                      votes_df.loc[votes_df['simulation_district']==i]['dem_votes'].sum()
                      , votes_df.loc[votes_df['simulation_district']==i]['rep_votes'].sum()
                      , votes_to_win]
    return district_votes 

In [5]:
test=district_totals(votes_2022_df, df)
test

,district,dem_votes,rep_votes,votes_to_win
1,1,98880,183261,141071
2,2,152441,153518,152980
3,3,134004,172927,153466
4,4,147075,169020,158048


In [12]:
test.iloc[1]['dem_votes']

152441

In [20]:
for i in range(4):
    if test.iloc[i]['dem_votes']>test.iloc[i]['rep_votes']:
        print('dem surplus =', test.iloc[i]['dem_votes']-test.iloc[i]['votes_to_win'], ', rep lost votes=', test.iloc[i]['rep_votes'])
    else:
        print('dem lost votes =', test.iloc[i]['dem_votes'], ', rep surplus votes=',  test.iloc[i]['rep_votes']-test.iloc[i]['votes_to_win'])

dem lost votes = 98880 , rep surplus votes= 42190
dem lost votes = 152441 , rep surplus votes= 538
dem lost votes = 134004 , rep surplus votes= 19461
dem lost votes = 147075 , rep surplus votes= 10972


In [ ]:
# input is datadrame with votes by party breakdown in each district with votes needed to win the district
# output is the wasted votes for democrats and republicans

def wasted_votes(district_votes):
    for i in range(4):
        if district_votes.iloc[i]['dem_votes']>district_votes.iloc[i]['rep_votes']:
            district_votes.iloc[i]['dem_wasted'] = district_votes.iloc[i]['dem_votes']-district_votes.iloc[i]['votes_to_win']
            # rep_wasted = rep_wasted + district_votes.loc[district_votes['district']==i]['rep_votes']
        else:
            district_votes.iloc[i]['dem_wasted'] = district_votes.iloc[i]['dem_votes']
            rep_wasted = rep_wasted + (district_votes.loc[district_votes['district']==i]['rep_votes']-district_votes.loc[district_votes['district']==i]['votes_to_win'])

        if votes_df.loc[votes_df['simulation_district']==i]['dem_votes'].sum()>votes_df.loc[votes_df['simulation_district']==i]['rep_votes'].sum():
            district_votes.loc[district_votes['district']==i,'party_winner']='D'
        else:
            district_votes.loc[district_votes['district']==i,'party_winner']='R'
    efficiency_gap = (rep_wasted - dem_wasted)/total_votes

In [ ]:
# compute EG for a given district map. Inputs are the union of district_totals and wasted_votes functions
# output is the efficiency gap as a percentage, rounded to 1 decimal place
def EG_func(votes_df, realloc_df):
    total_votes = votes_df['total_votes'].sum()